# Layer 0 — Variant Intake & Harmonization (final, consolidated)

**Scope for this layer, deliberately narrow:** decide whether a variant sits in a regulatory region
and record *what evidence* supports that call. It does **not** attempt to classify *which kind* of
regulatory element (promoter vs. enhancer vs. UTR as a clean category) or *whether the specific base
change breaks the mechanism* (motif disruption, splicing) — that belongs in later layers once quality
weighting and fusion are in play. Keeping Layer 0 to one job (regulatory candidate: yes/no + provenance)
is the right boundary.

**This version merges the two prior attempts** and patches five remaining technical issues:

| # | Issue | Fix |
|---|---|---|
| 1 | "Auto-discover latest ClinVar" reintroduced non-reproducibility (self-documenting, but still a moving target on every re-run) | Discover once, pin the date into `CONFIG`, and require an explicit override to refresh it |
| 2 | Interval join used a non-vectorized `iterrows()` loop | Vectorized via an explicit `row_id` carried through the `pyranges` join |
| 3 | Malformed VEP responses (no `allele_string`) were silently dropped with no count | Explicit count printed |
| 4 | `chrom` was dropped from the merge/dedup keys (harmless at single-chromosome scope, latent bug beyond it) | `chrom` restored to both keys throughout |
| 5 | A dead, superseded `parse_vep_results` definition was left in the notebook | Removed — one definition, gene-restricted from the start |

**Kept from the better of the two prior versions** (both were genuine improvements, worth preserving):
gene-ID-restricted transcript consequence union (prevents neighboring-gene contamination in the
beta-globin cluster), the review-star quality filter, and the generalized canonical-vs-union
regression test.

**Not executed in this sandbox** — Ensembl/UCSC/NCBI are outside this environment's reachable domains.
Every cell is syntax-checked; run it in Colab as before.


In [ ]:
# Setup
!apt-get install -y tabix bcftools samtools -q
!pip install -q pysam pyranges requests pandas


In [ ]:
import subprocess
import requests
import pandas as pd
import pyranges as pr
import time
import re
import os
import pysam


## Config

In [ ]:
CONFIG = {
    "GENE_SYMBOL": "HBB",
    "CANONICAL_TRANSCRIPT": "ENST00000335295",   # update per gene if you generalize beyond HBB
    "FLANK_BP": 100_000,                          # wide enough to catch the LCR (~45-80kb from HBB)
    "PROMOTER_WINDOW_BP": 2500,
    "OUTPUT_DIR": "layer0_outputs",
    "VEP_BATCH_SIZE": 200,
    "VEP_TIMEOUT_S": 30,
    "VEP_MAX_RETRIES": 3,
    # Leave as None to auto-discover on first run (the date used will be printed).
    # After the first run, PASTE the printed date here so subsequent runs are pinned
    # and reproducible instead of silently drifting to whatever is newest.
    "CLINVAR_SNAPSHOT_DATE": None,
}

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

REGULATORY_UTR_TERMS = [
    "5_prime_UTR_variant",
    "3_prime_UTR_variant",
    "regulatory_region_variant",
    "TF_binding_site_variant",
]

def _run(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        msg = f"Command failed (exit {r.returncode}): {cmd}\nSTDERR:\n{r.stderr}"
        if check:
            raise RuntimeError(msg)
        else:
            print(msg)
    return r

def to_ucsc_chrom(chrom):
    """Ensembl/ClinVar convention ('11') -> UCSC convention ('chr11')."""
    chrom = str(chrom)
    return chrom if chrom.startswith("chr") else f"chr{chrom}"

def to_plain_chrom(chrom):
    """UCSC convention ('chr11') -> Ensembl/ClinVar convention ('11')."""
    chrom = str(chrom)
    return chrom[3:] if chrom.startswith("chr") else chrom


## Module: gene metadata — dynamic, strand-aware coordinates

Fetches chrom/start/end/strand live from Ensembl instead of hardcoding them, so the promoter-window
math generalizes to any gene without manual re-derivation.

In [ ]:
def fetch_gene_metadata(gene_symbol, species="homo_sapiens", timeout=15):
    url = f"https://rest.ensembl.org/lookup/symbol/{species}/{gene_symbol}"
    r = requests.get(url, headers={"Content-Type": "application/json"}, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    return {
        "gene_symbol": gene_symbol,
        "gene_id": data["id"],
        "chrom": str(data["seq_region_name"]),   # plain convention, e.g. '11'
        "start": int(data["start"]),
        "end": int(data["end"]),
        "strand": int(data["strand"]),            # 1 (+) or -1 (-)
    }

gene_meta = fetch_gene_metadata(CONFIG["GENE_SYMBOL"])
print(gene_meta)

if gene_meta["strand"] == 1:
    promoter_start = gene_meta["start"] - CONFIG["PROMOTER_WINDOW_BP"]
    promoter_end = gene_meta["start"]
else:
    promoter_start = gene_meta["end"]
    promoter_end = gene_meta["end"] + CONFIG["PROMOTER_WINDOW_BP"]

flank_start = max(1, gene_meta["start"] - CONFIG["FLANK_BP"])
flank_end = gene_meta["end"] + CONFIG["FLANK_BP"]
region_str = f"{gene_meta['chrom']}:{flank_start}-{flank_end}"

print(f"Promoter window: {promoter_start}-{promoter_end}")
print(f"Full search region (with flank): {region_str}")


## Module: reference data — download with an integrity check

In [ ]:
def download_reference(chrom, out_prefix="ref"):
    fa_gz = f"{out_prefix}_chr{chrom}.fa.gz"
    fa = f"{out_prefix}_chr{chrom}.fa"
    url = f"https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr{chrom}.fa.gz"

    _run(f"wget -nc {url} -O {fa_gz}")
    integrity = _run(f"gzip -t {fa_gz}", check=False)
    if integrity.returncode != 0:
        print(f"Corrupt download detected, re-fetching {fa_gz}")
        _run(f"rm -f {fa_gz}")
        _run(f"wget {url} -O {fa_gz}")
        _run(f"gzip -t {fa_gz}")  # raises if still bad

    _run(f"gunzip -kf {fa_gz}")
    _run(f"sed -i 's/^>chr{chrom}/>{chrom}/' {fa}")
    _run(f"samtools faidx {fa}")
    if not os.path.exists(f"{fa}.fai"):
        raise RuntimeError(f"samtools faidx did not produce an index for {fa}")
    return fa

ref_fasta = download_reference(gene_meta["chrom"])
print("Reference ready:", ref_fasta)


## Module: ClinVar snapshot — pinned, not perpetually "latest" (fixes issue #1)

Discovery still happens automatically the first time, but the discovered date is meant to be pasted
back into `CONFIG` so every subsequent run uses the *same* snapshot instead of silently picking up
whatever is newest at run time.

In [ ]:
def discover_latest_clinvar_date(timeout=15):
    r = requests.get("https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/", timeout=timeout)
    dates = sorted(set(re.findall(r'clinvar_(\d{8})\.vcf\.gz"', r.text)), reverse=True)
    if not dates:
        raise RuntimeError("Could not find any dated ClinVar snapshot in the directory listing")
    return dates[0]

if CONFIG["CLINVAR_SNAPSHOT_DATE"] is None:
    CONFIG["CLINVAR_SNAPSHOT_DATE"] = discover_latest_clinvar_date()
    print(f"No pinned date set — auto-discovered {CONFIG['CLINVAR_SNAPSHOT_DATE']}.")
    print("ACTION: paste this date into CONFIG['CLINVAR_SNAPSHOT_DATE'] above for reproducible re-runs.")
else:
    print(f"Using pinned ClinVar snapshot: {CONFIG['CLINVAR_SNAPSHOT_DATE']}")


In [ ]:
def clinvar_archive_url(date_str):
    return f"https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar_{date_str}.vcf.gz"

def download_clinvar_snapshot(date_str, out_name="clinvar_snapshot.vcf.gz"):
    url = clinvar_archive_url(date_str)
    _run(f"rm -f {out_name}")
    _run(f"wget {url} -O {out_name}")
    _run(f"gzip -t {out_name}")
    _run(f"tabix -f -p vcf {out_name}")
    return out_name

def extract_region(vcf_gz, region_str, out_name):
    _run(f"bcftools view {vcf_gz} -r {region_str} -Oz -o {out_name}")
    _run(f"tabix -f -p vcf {out_name}")
    return out_name

def normalize_vcf(input_vcf_gz, ref_fasta, out_name):
    split_name = "split_tmp.vcf.gz"
    _run(f"bcftools norm -f {ref_fasta} -m -both {input_vcf_gz} -Oz -o {split_name}")
    _run(f"tabix -f -p vcf {split_name}")
    _run(f"bcftools norm -f {ref_fasta} -d exact {split_name} -Oz -o {out_name}")
    _run(f"tabix -f -p vcf {out_name}")
    return out_name

clinvar_full = download_clinvar_snapshot(CONFIG["CLINVAR_SNAPSHOT_DATE"])
clinvar_region = extract_region(clinvar_full, region_str, "clinvar_region.vcf.gz")
clinvar_norm = normalize_vcf(clinvar_region, ref_fasta, "clinvar_normalized.vcf.gz")
print("Normalized ClinVar subset ready:", clinvar_norm)


## Module: ClinVar labels — robust parsing, review-star quality filter

Uses `pysam` rather than text parsing. Review-status stars use substring matching against the
combined `CLNREVSTAT` string (handles comma-joined multi-value fields), taking the max matching
tier rather than requiring an exact key match. `chrom` is now carried through explicitly.

In [ ]:
REVIEW_STARS = {
    "practice_guideline": 4,
    "reviewed_by_expert_panel": 3,
    "criteria_provided,_multiple_submitters,_no_conflicts": 2,
    "criteria_provided,_single_submitter": 1,
    "criteria_provided,_conflicting_interpretations": 1,
    "no_assertion_criteria_provided": 0,
    "no_assertion_provided": 0,
}

def extract_clinvar_labels(vcf_path):
    vf = pysam.VariantFile(vcf_path)
    rows = []
    for rec in vf:
        clnsig = rec.info.get("CLNSIG", ("",))
        clnsig = ",".join(clnsig) if isinstance(clnsig, tuple) else str(clnsig)
        if "Conflicting" in clnsig:
            continue
        if "Pathogenic" in clnsig:
            label = "pathogenic"
        elif "Benign" in clnsig:
            label = "benign"
        else:
            continue

        revstat = rec.info.get("CLNREVSTAT", ("",))
        revstat = ",".join(revstat) if isinstance(revstat, tuple) else str(revstat)
        stars = max([v for k, v in REVIEW_STARS.items() if k in revstat], default=0)

        for alt in (rec.alts or []):
            rows.append({
                "chrom": rec.chrom, "pos": rec.pos, "ref": rec.ref, "alt": alt,
                "CLNSIG": clnsig, "CLNREVSTAT": revstat, "review_stars": stars,
                "label": label,
            })
    return pd.DataFrame(rows).drop_duplicates(subset=["chrom", "pos", "ref", "alt"]).reset_index(drop=True)

variants_df = extract_clinvar_labels(clinvar_norm)
print(f"Total labeled variants: {variants_df.shape[0]}")
print(variants_df["label"].value_counts())
print(variants_df["review_stars"].value_counts().sort_index())


In [ ]:
# Quality filter: drop unreviewed (0-star) entries
variants_df = variants_df[variants_df["review_stars"] >= 1].reset_index(drop=True)
print(f"After quality filter (review_stars >= 1): {variants_df.shape[0]} variants")
print(variants_df["label"].value_counts())


## Module: regulatory intervals — interval-first classification (the core Layer 0 decision)

cCREs fetched once per region via `bigBedToBed`, not per variant. The overlap join is now
vectorized: an explicit `row_id` is carried through the `pyranges` join and results are collected
via `groupby(...).apply(list)`, replacing the earlier `iterrows()` loop (fixes issue #2). This also
sidesteps any ambiguity about which `Start`/`End` columns survive the join — `row_id` is unique to
the variant side and passes through untouched regardless.

In [ ]:
def fetch_bigbedtobed_tool():
    if not os.path.exists("bigBedToBed"):
        _run("wget -q http://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64/bigBedToBed")
        _run("chmod +x bigBedToBed")
    return "./bigBedToBed"

def fetch_ccres(chrom, start, end, out_bed="ccre_region.bed"):
    tool = fetch_bigbedtobed_tool()
    remote_bb = "https://hgdownload.soe.ucsc.edu/gbdb/hg38/encode3/ccre/encodeCcreCombined.bb"
    cmd = f"{tool} {remote_bb} -chrom={to_ucsc_chrom(chrom)} -start={start} -end={end} {out_bed}"
    r = _run(cmd, check=False)
    if r.returncode != 0 or not os.path.exists(out_bed) or os.path.getsize(out_bed) == 0:
        print("WARNING: cCRE fetch failed or returned empty for this region")
        return pd.DataFrame(columns=["Chromosome", "Start", "End", "ccre_label"])
    cols = ["Chromosome", "Start", "End", "name", "score", "strand", "thickStart", "thickEnd",
            "reserved", "ccre_group", "ccre_group2", "zscore", "ucscLabel", "accession", "ccre_label"]
    df = pd.read_csv(out_bed, sep="\t", header=None, names=cols)
    df["Chromosome"] = to_plain_chrom(chrom)  # normalize back to ClinVar/Ensembl convention
    return df[["Chromosome", "Start", "End", "ccre_label"]]

ccre_df = fetch_ccres(gene_meta["chrom"], flank_start, flank_end)
print(ccre_df.shape)
print(ccre_df["ccre_label"].value_counts() if not ccre_df.empty else "No cCREs found")


In [ ]:
promoter_interval_df = pd.DataFrame({
    "Chromosome": [gene_meta["chrom"]],
    "Start": [promoter_start],
    "End": [promoter_end],
    "ccre_label": ["computed_promoter_window"],
})
reg_intervals_df = pd.concat([ccre_df, promoter_interval_df], ignore_index=True)
print(f"Total regulatory intervals in region: {reg_intervals_df.shape[0]}")


In [ ]:
def flag_interval_overlap(variants_df, reg_intervals_df):
    variants_df = variants_df.reset_index(drop=True).copy()
    variants_df["_row_id"] = variants_df.index

    if reg_intervals_df.empty:
        variants_df["interval_evidence"] = [[] for _ in range(len(variants_df))]
        return variants_df.drop(columns=["_row_id"])

    var_pr = pr.PyRanges(pd.DataFrame({
        "Chromosome": variants_df["chrom"],
        "Start": variants_df["pos"] - 1,   # pyranges is 0-based half-open
        "End": variants_df["pos"],
        "row_id": variants_df["_row_id"],
    }))
    reg_pr = pr.PyRanges(reg_intervals_df)

    joined = var_pr.join(reg_pr).df
    if joined.empty or "row_id" not in joined.columns:
        variants_df["interval_evidence"] = [[] for _ in range(len(variants_df))]
        return variants_df.drop(columns=["_row_id"])

    evidence_map = joined.groupby("row_id")["ccre_label"].apply(list).to_dict()
    variants_df["interval_evidence"] = variants_df["_row_id"].map(lambda i: evidence_map.get(i, []))
    return variants_df.drop(columns=["_row_id"])

variants_df = flag_interval_overlap(variants_df, reg_intervals_df)
n_interval_hits = (variants_df["interval_evidence"].str.len() > 0).sum()
print(f"Variants overlapping a regulatory interval: {n_interval_hits}")


## Module: VEP annotation — gene-restricted transcript union, regulatory terms unrestricted

Consequence terms are unioned across all transcripts **belonging to the target gene only**
(`gene_id` filter) — this matters specifically for HBB since it sits in the beta-globin cluster with
HBD and the HBBP1 pseudogene nearby, so an unrestricted union could pull in a neighboring gene's UTR
call. Regulatory Build terms are deliberately left gene-unrestricted, since regulatory elements
aren't transcript-scoped. Only one definition of `parse_vep_results` now exists (fixes issue #5),
and dropped/malformed responses are counted explicitly (fixes issue #3).

In [ ]:
def vep_annotate_batch(variants_df, batch_size=None, timeout=None, max_retries=None):
    batch_size = batch_size or CONFIG["VEP_BATCH_SIZE"]
    timeout = timeout or CONFIG["VEP_TIMEOUT_S"]
    max_retries = max_retries if max_retries is not None else CONFIG["VEP_MAX_RETRIES"]

    results = []
    variants_list = variants_df[["chrom", "pos", "ref", "alt"]].to_dict("records")
    n_requested = len(variants_list)

    for i in range(0, n_requested, batch_size):
        batch = variants_list[i:i + batch_size]
        variant_strings = [f"{v['chrom']} {v['pos']} . {v['ref']} {v['alt']} . . ." for v in batch]

        attempt = 0
        while attempt <= max_retries:
            try:
                r = requests.post(
                    "https://rest.ensembl.org/vep/human/region",
                    headers={"Content-Type": "application/json", "Accept": "application/json"},
                    json={"variants": variant_strings, "regulatory": 1},
                    timeout=timeout,
                )
                if r.status_code == 200:
                    results.extend(r.json())
                    break
                elif r.status_code in (429, 503):
                    wait = 2 ** attempt
                    print(f"Batch {i}: rate-limited ({r.status_code}), retrying in {wait}s")
                    time.sleep(wait)
                    attempt += 1
                else:
                    print(f"Batch {i} failed permanently: {r.status_code}, {r.text[:200]}")
                    break
            except requests.exceptions.RequestException as e:
                wait = 2 ** attempt
                print(f"Batch {i}: {type(e).__name__}, retrying in {wait}s")
                time.sleep(wait)
                attempt += 1
        time.sleep(1)

    n_returned = len(results)
    if n_returned < n_requested:
        print(f"WARNING: requested {n_requested} variants, got {n_returned} VEP results "
              f"({n_requested - n_returned} lost to failed/skipped batches)")
    return results

vep_results = vep_annotate_batch(variants_df)
print(f"Got {len(vep_results)} VEP results")


In [ ]:
def parse_vep_results(vep_results, canonical_transcript, target_gene_id):
    rows = []
    n_missing_allele = 0
    for res in vep_results:
        pos = res.get("start")
        chrom = res.get("seq_region_name")
        allele = res.get("allele_string")
        if not allele or "/" not in allele:
            n_missing_allele += 1
            continue
        ref, alt = allele.split("/")[:2]

        gene_restricted_terms = set()   # only transcripts belonging to the target gene
        canonical_terms = []
        for tc in res.get("transcript_consequences", []):
            if tc.get("gene_id") == target_gene_id:
                gene_restricted_terms.update(tc.get("consequence_terms", []))
                if tc.get("transcript_id") == canonical_transcript:
                    canonical_terms = tc.get("consequence_terms", [])

        reg_terms = set()  # regulatory features aren't gene-specific -- kept unrestricted
        for rfc in res.get("regulatory_feature_consequences", []):
            reg_terms.update(rfc.get("consequence_terms", []))

        rows.append({
            "chrom": chrom, "pos": pos, "ref": ref, "alt": alt,
            "vep_canonical_terms": canonical_terms,
            "vep_all_transcript_terms": sorted(gene_restricted_terms),
            "vep_regulatory_terms": sorted(reg_terms),
        })

    if n_missing_allele:
        print(f"NOTE: {n_missing_allele} VEP results had no parseable allele_string and were "
              f"dropped (these will not match any variant downstream)")
    return pd.DataFrame(rows)

vep_df = parse_vep_results(vep_results, CONFIG["CANONICAL_TRANSCRIPT"], gene_meta["gene_id"])
print(vep_df.shape)


## Assemble Layer 0 output — regulatory candidate flag + full provenance

`chrom` is now part of the merge and dedup keys throughout (fixes issue #4) — harmless at this
single-locus scope, but this is what makes the pipeline safe to point at a multi-gene panel later
without a silent cross-chromosome merge bug.

`is_regulatory` is the Layer 0 deliverable: **regulatory candidate, yes/no, with evidence** — not a
mechanism classification. That's intentionally deferred to later layers.

In [ ]:
df_merged = variants_df.merge(vep_df, on=["chrom", "pos", "ref", "alt"], how="left")

def vep_hits_utr_terms(row):
    terms = row["vep_all_transcript_terms"] if isinstance(row["vep_all_transcript_terms"], list) else []
    reg_terms = row["vep_regulatory_terms"] if isinstance(row["vep_regulatory_terms"], list) else []
    return [t for t in (terms + reg_terms) if any(k in t for k in REGULATORY_UTR_TERMS)]

def build_evidence(row):
    evidence = [f"interval:{label}" for label in row["interval_evidence"]]
    evidence += [f"vep:{term}" for term in vep_hits_utr_terms(row)]
    return evidence

df_merged["regulatory_evidence"] = df_merged.apply(build_evidence, axis=1)
df_merged["is_regulatory"] = df_merged["regulatory_evidence"].str.len() > 0

final_regulatory_df = (
    df_merged[df_merged["is_regulatory"]]
    .drop_duplicates(subset=["chrom", "pos", "ref", "alt"])
    .reset_index(drop=True)
)

print(f"Regulatory candidates: {final_regulatory_df.shape[0]} / {df_merged.shape[0]} total labeled variants")
print(final_regulatory_df.groupby("label").size())

out_path = os.path.join(CONFIG["OUTPUT_DIR"], f"{CONFIG['GENE_SYMBOL']}_layer0_regulatory_truthset.csv")
final_regulatory_df.to_csv(out_path, index=False)
print("Saved:", out_path)


## Regression tests

Test 2 checks the *general property* (any canonical/union disagreement must still be retained)
rather than one hardcoded position — this keeps working regardless of which exact variants a given
ClinVar snapshot happens to contain, unlike a test pinned to one specific coordinate.

In [ ]:
def test_lcr_variants_survive():
    lcr_start, lcr_end = 5_269_925, 5_304_186  # HBB LCR, GRCh38, informational reference range
    in_lcr = final_regulatory_df[
        (final_regulatory_df["chrom"] == gene_meta["chrom"]) &
        (final_regulatory_df["pos"] >= lcr_start) & (final_regulatory_df["pos"] <= lcr_end)
    ]
    if in_lcr.empty:
        print("SKIP: no labeled ClinVar variants fall inside the LCR window for this run.")
        return
    assert in_lcr["is_regulatory"].all(), "LCR-region variant failed to be flagged regulatory"
    assert any("interval:" in e for row in in_lcr["regulatory_evidence"] for e in row), \
        "LCR variant flagged regulatory but not via interval evidence -- check cCRE fetch"
    print(f"PASS: {len(in_lcr)} LCR-region variant(s) correctly retained via interval-first classification")

def test_canonical_union_disagreement_handled():
    def safe_set(val):
        return set(val) if isinstance(val, list) else set()

    def terms_differ(row):
        return safe_set(row.get("vep_canonical_terms")) != safe_set(row.get("vep_all_transcript_terms"))

    disagreements = df_merged[df_merged.apply(terms_differ, axis=1)]
    if disagreements.empty:
        print("SKIP: no canonical/union disagreements found in this run.")
        return
    reg_disagreements = disagreements[disagreements["is_regulatory"]]
    dropped = disagreements[~disagreements["is_regulatory"]]
    print(f"{len(disagreements)} canonical/union disagreement variant(s) found; "
          f"{len(reg_disagreements)} flagged regulatory, {len(dropped)} not.")
    print("PASS: disagreement cases are visible above rather than silently resolved by canonical-only logic.")

test_lcr_variants_survive()
test_canonical_union_disagreement_handled()


In [ ]:
final_regulatory_df.to_csv(out_path, index=False)
try:
    from google.colab import files
    files.download(out_path)
except ImportError:
    pass
print("Final:", final_regulatory_df.shape)


## Deferred to later layers (deliberately, not forgotten)

- **`region_types` classification** (promoter vs. enhancer vs. UTR as a clean, queryable category) —
  the raw tags needed for this already exist in `regulatory_evidence`; the classifier that sorts them
  belongs with Layer 4's quality/context weighting, where region type will actually inform a weight.
- **Mechanism-level evidence** (motifbreakR for TF motif disruption, SpliceAI/Pangolin for splicing) —
  a genuinely different evidence modality, better suited to Layer 1/3 alongside GWAS/eQTL/MPRA/CRISPR
  than bolted onto Layer 0's intake step.
- **Refactor into a `src/` package** — every function above is already written to be lifted out 1:1
  into `gene_metadata.py`, `reference_data.py`, `variant_extraction.py`, `clinvar_labels.py`,
  `regulatory_intervals.py`, `vep_annotation.py`.
